In [28]:
import torch
from diffusers.models.attention import Attention

在Attention()使用的时候，如果query,key,value都是一样的，那么就是自注意力
如果query和key/value不一样，那么就是交叉注意力
还有一点，在使用Attention初始化的时候是要确定AttnProcessor(注意力过程，不指定的话就是AttnProcessor2_0过程)

理一下整体的过程：首先是拿到query/key/value 将其维度变成num_attention_heads*attention_head_dim
然后按照注意力过程做注意力过程，最后把输出结果投影到query的维度

下面是一个简单的Attention()类初始化的方式(自注意力):
在不指定cross_attention_dim(key/value)的维度时，会自动将query_dim赋值给cross_attention_dim

In [29]:
query_dim,cross_attention_dim,num_attention_heads,attention_head_dim,dropout,attention_bias,upcast_attention=128,256,8,64,0.0,False,False
self_attention_layer=Attention(query_dim=query_dim,#query的编码维度
                          heads=num_attention_heads, #注意力头的数量
                          dim_head=attention_head_dim, #每个注意力头的维度
                          dropout=dropout,
                          bias=attention_bias, #是否使用注意力偏置
                          upcast_attention=upcast_attention, #是否升级到float32类型
                          )

print(self_attention_layer)


Attention(
  (to_q): LoRACompatibleLinear(in_features=128, out_features=512, bias=False)
  (to_k): LoRACompatibleLinear(in_features=128, out_features=512, bias=False)
  (to_v): LoRACompatibleLinear(in_features=128, out_features=512, bias=False)
  (to_out): ModuleList(
    (0): LoRACompatibleLinear(in_features=512, out_features=128, bias=True)
    (1): Dropout(p=0.0, inplace=False)
  )
)


In [30]:
batch_size,len_sequence=64,32
query=torch.randn(size=[batch_size,len_sequence,query_dim])

In [32]:
self_out_put=self_attention_layer(query,attention_mask=None) #自注意力直接将query赋值给key/value
print(self_out_put.shape)

torch.Size([64, 32, 128])


下面是交叉注意力初始化的简单示例：

In [33]:
cross_attention_layers=Attention(
                          query_dim=query_dim,#query的编码维度
                          cross_attention_dim=cross_attention_dim, #key/value的编码长度
                          heads=num_attention_heads, #注意力头的数量
                          dim_head=attention_head_dim, #每个注意力头的维度
                          dropout=dropout,
                          bias=attention_bias, #是否使用注意力偏置
                          upcast_attention=upcast_attention, #是否升级到float32类型 
                         )
print(cross_attention_layers)

Attention(
  (to_q): LoRACompatibleLinear(in_features=128, out_features=512, bias=False)
  (to_k): LoRACompatibleLinear(in_features=256, out_features=512, bias=False)
  (to_v): LoRACompatibleLinear(in_features=256, out_features=512, bias=False)
  (to_out): ModuleList(
    (0): LoRACompatibleLinear(in_features=512, out_features=128, bias=True)
    (1): Dropout(p=0.0, inplace=False)
  )
)


In [34]:
key_1=torch.randn(size=[batch_size,16,cross_attention_dim])
cross_out_put=cross_attention_layers(query,key_1,)
print(cross_out_put.shape)

torch.Size([64, 32, 128])


Attention()就先理解到这里